In [ ]:
import requests

def check_urls(url_list):
    """
    Checks the status of a list of URLs using HEAD requests.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    print("Checking URLs...")
    for url in url_list:
        try:
            # Use a HEAD request for efficiency, timeout after 5 seconds
            response = requests.head(url, headers=headers, timeout=5, allow_redirects=True)
            
            # Check if the status code is in the 2xx (successful) or 3xx (redirection) range
            if response.ok:
                print(f"✅ SUCCESS: {url} (Status: {response.status_code})")
            else:
                print(f"❌ FAILED:  {url} (Status: {response.status_code})")

        except requests.exceptions.RequestException as e:
            # Handle connection errors, timeouts, etc.
            print(f"‼️ ERROR:   {url} (Could not connect: {e.__class__.__name__})")

# The list of URLs to check
urls_to_test = [
    # Authentication
    "https://stripe.com/docs/keys",
    "https://stripe.com/docs/api/authentication",
    
    # Error handling
    "https://stripe.com/docs/api/errors",
    "https://stripe.com/docs/error-codes",
    
    # Webhooks
    "https://stripe.com/docs/webhooks",
    "https://stripe.com/docs/webhooks/signatures",
    "https://stripe.com/docs/webhooks/test",
    
    # Payments
    "https://stripe.com/docs/payments",
    "https://stripe.com/docs/payments/accept-a-payment",
    "https://stripe.com/docs/payments/payment-intents",
    
    # API
    "https://stripe.com/docs/api/idempotent_requests",
    "https://stripe.com/docs/api/versioning",
    "https://docs.stripe.com/rate-limits",
    
    # Common issues
    "https://stripe.com/docs/testing",
    "https://stripe.com/docs/disputes",
]

# Run the check
if __name__ == "__main__":
    check_urls(urls_to_test)

Checking URLs...
✅ SUCCESS: https://stripe.com/docs/keys (Status: 200)
✅ SUCCESS: https://stripe.com/docs/api/authentication (Status: 200)
✅ SUCCESS: https://stripe.com/docs/api/errors (Status: 200)
✅ SUCCESS: https://stripe.com/docs/error-codes (Status: 200)
✅ SUCCESS: https://stripe.com/docs/webhooks (Status: 200)
✅ SUCCESS: https://stripe.com/docs/webhooks/signatures (Status: 200)
✅ SUCCESS: https://stripe.com/docs/webhooks/test (Status: 200)
✅ SUCCESS: https://stripe.com/docs/payments (Status: 200)
✅ SUCCESS: https://stripe.com/docs/payments/accept-a-payment (Status: 200)
✅ SUCCESS: https://stripe.com/docs/payments/payment-intents (Status: 200)
✅ SUCCESS: https://stripe.com/docs/api/idempotent_requests (Status: 200)
‼️ ERROR:   https://stripe.com/docs/api/versioning (Could not connect: ReadTimeout)
❌ FAILED:  https://stripe.com/docs/api/rate_limits (Status: 404)
‼️ ERROR:   https://stripe.com/docs/testing (Could not connect: ReadTimeout)
‼️ ERROR:   https://stripe.com/docs/disputes

In [1]:
from decouple import config
import requests

API_URL = "https://router.huggingface.co/hf-inference/models/facebook/bart-large-mnli"
headers = {
    "Authorization": f"Bearer {config('HF_TOKEN')}",
}

def query(payload):
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

output = query({
    "inputs": "Hi, I recently bought a device from your company but it is not working as advertised and I would like to get reimbursed!",
    "parameters": {"candidate_labels": ["refund", "legal", "faq"]},
})

In [2]:
output

[{'label': 'refund', 'score': 0.8777873516082764},
 {'label': 'faq', 'score': 0.10522671788930893},
 {'label': 'legal', 'score': 0.016985908150672913}]

In [3]:
from decouple import config
import requests

API_URL = "https://router.huggingface.co/hf-inference/models/facebook/bart-large-mnli"
headers = {
    "Authorization": f"Bearer {config('HF_TOKEN')}",
}

ISSUE_LABELS   = ["billing", "technical", "account", "feature", "general"]
URGENCY_LABELS = ["high", "medium", "low"]

def query(payload):
    response = requests.post(API_URL, headers=headers, json=payload)
    response.raise_for_status()
    return response.json()

def classify(ticket_text: str) -> dict:
    """
    Classifies a support ticket into issue type and urgency.

    Returns:
        {
            "issue_type": "billing",
            "urgency": "high",
            "issue_score": 0.91,
            "urgency_score": 0.78
        }
    """
    issue_output = query({
        "inputs": ticket_text,
        "parameters": {"candidate_labels": ISSUE_LABELS}
    })

    urgency_output = query({
        "inputs": ticket_text,
        "parameters": {"candidate_labels": URGENCY_LABELS}
    })

    return {
        "issue_type":    issue_output[0]["label"],
        "urgency":       urgency_output[0]["label"],
        "issue_score":   round(issue_output[0]["score"], 4),
        "urgency_score": round(urgency_output[0]["score"], 4),
    }

In [4]:
ticket = "I was charged twice for my subscription this month and need a refund immediately."
result = classify(ticket)
print(result)
# {'issue_type': 'billing', 'urgency': 'high', 'issue_score': 0.9412, 'urgency_score': 0.8231}

{'issue_type': 'billing', 'urgency': 'high', 'issue_score': 0.5214, 'urgency_score': 0.7167}


In [2]:
from decouple import config
import json
import requests

# Load API credentials from environment variables (ensure these are set)
ZENDESK_API_TOKEN = config('ZENDESK_API_TOKEN')
ZENDESK_USER_EMAIL = config('ZENDESK_USER_EMAIL')
ZENDESK_SUBDOMAIN = config('ZENDESK_SUBDOMAIN')

if not ZENDESK_API_TOKEN or not ZENDESK_USER_EMAIL or not ZENDESK_SUBDOMAIN:
    print('Error: Missing required environment variables.')
    exit(1)

# Ticket to update
ticket_id = '1'
body = 'Thanks for choosing Acme Jet Motors.'

# Build JSON data with public comment
data = {
    'ticket': {
        'comment': {
            'body': body,
            'public': True
        }
    }
}

url = f'https://{ZENDESK_SUBDOMAIN}.zendesk.com/api/v2/tickets/{ticket_id}.json'
auth = (f"{ZENDESK_USER_EMAIL}/token", ZENDESK_API_TOKEN)

# Serialize data to JSON string
payload = json.dumps(data)

headers = {
    'Content-Type': 'application/json'
}

try:
    # Pass serialized JSON payload with headers
    response = requests.put(url, data=payload, headers=headers, auth=auth)
    response.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")
    exit(1)

print(f"Successfully added comment to ticket #{ticket_id}")
print("Response:", response.json())

Successfully added comment to ticket #1
Response: {'ticket': {'url': 'https://none-55378.zendesk.com/api/v2/tickets/1.json', 'id': 1, 'external_id': None, 'via': {'channel': 'sample_ticket', 'source': {'from': {}, 'to': {}, 'rel': None}}, 'created_at': '2026-03-11T19:13:52Z', 'updated_at': '2026-03-12T15:45:58Z', 'generated_timestamp': 1773256456, 'type': None, 'subject': 'SAMPLE: How does Zendesk work', 'raw_subject': 'SAMPLE: How does Zendesk work', 'description': "Hello, let's see how you or your agents can easily respond to and solve tickets.\n\nFeel free to email additional customer test inquiries to **support@none-55378.zendesk.com**.\n\nBut first, let's start by solving one ticket.\n\nYour Zendesk Team", 'priority': 'normal', 'status': 'pending', 'recipient': None, 'requester_id': 34566765265565, 'submitter_id': 34566765265565, 'assignee_id': 34512710018717, 'organization_id': None, 'group_id': 34512695547421, 'collaborator_ids': [34512710018717], 'follower_ids': [], 'email_cc_i

In [3]:
import requests
from decouple import config
import json

# Load the API credentials from required environment variables (ensure these are set)
ZENDESK_API_TOKEN = config('ZENDESK_API_TOKEN')
ZENDESK_USER_EMAIL = config('ZENDESK_USER_EMAIL')
ZENDESK_SUBDOMAIN = config('ZENDESK_SUBDOMAIN')

if not ZENDESK_API_TOKEN or not ZENDESK_USER_EMAIL or not ZENDESK_SUBDOMAIN:
    print('Error: Missing required environment variables.')
    exit(1)

# New ticket info
subject = 'My printer is on fire!'
body = 'The smoke is very colorful.'

# Package the data in a dictionary matching the expected JSON
data = {
    'ticket': {
        'subject': subject,
        'comment': {
            'body': body,
            'public': True
        }
    }
}

url = f'https://{ZENDESK_SUBDOMAIN}.zendesk.com/api/v2/tickets.json'

auth = f'{ZENDESK_USER_EMAIL}/token', ZENDESK_API_TOKEN
# Manually serialize the data dictionary to a JSON string
payload = json.dumps(data)

headers = {
    'Content-Type': 'application/json'
}

# Do the HTTP post request with serialized JSON payload and headers
response = requests.post(url, data=payload, headers=headers, auth=auth)

# Check for HTTP codes other than 201 (Created)
if response.status_code != 201:
    print('Status:', response.status_code, 'Problem with the request. Exiting.')
    exit()

# Report success
print('Successfully created the ticket.')
print(response.json())


Successfully created the ticket.
{'ticket': {'url': 'https://none-55378.zendesk.com/api/v2/tickets/2.json', 'id': 2, 'external_id': None, 'via': {'channel': 'api', 'source': {'from': {}, 'to': {}, 'rel': None}}, 'created_at': '2026-03-12T15:52:17Z', 'updated_at': '2026-03-12T15:52:17Z', 'generated_timestamp': 0, 'type': None, 'subject': 'My printer is on fire!', 'raw_subject': 'My printer is on fire!', 'description': 'The smoke is very colorful.', 'priority': 'normal', 'status': 'open', 'recipient': None, 'requester_id': 34512710018717, 'submitter_id': 34512710018717, 'assignee_id': 34512710018717, 'organization_id': 34512695548189, 'group_id': 34512695547421, 'collaborator_ids': [], 'follower_ids': [], 'email_cc_ids': [], 'forum_topic_id': None, 'problem_id': None, 'has_incidents': False, 'is_public': True, 'due_at': None, 'tags': [], 'custom_fields': [{'id': 34512681273757, 'value': None}, {'id': 34512696148253, 'value': None}, {'id': 34512696152093, 'value': None}, {'id': 3451271047

In [2]:
from sentence_transformers import SentenceTransformer
sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(sentences)
print(embeddings)


'[Errno -2] Name or service not known' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


# Sentences we want sentence embeddings for
sentences = ['This is an example sentence', 'Each sentence is converted']

# Load model from HuggingFace Hub
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

# Tokenize sentences
encoded_input = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

# Compute token embeddings
with torch.no_grad():
    model_output = model(**encoded_input)

# Perform pooling
sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

# Normalize embeddings
sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

print("Sentence embeddings:")
print(sentence_embeddings)


/home/praizdev/Documents/back-end/ai/stripe-support-ai/backend/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3267.60it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence embeddings:
tensor([[ 6.7657e-02,  6.3496e-02,  4.8713e-02,  7.9305e-02,  3.7448e-02,
          2.6528e-03,  3.9375e-02, -7.0985e-03,  5.9361e-02,  3.1537e-02,
          6.0098e-02, -5.2905e-02,  4.0607e-02, -2.5931e-02,  2.9843e-02,
          1.1269e-03,  7.3515e-02, -5.0382e-02, -1.2239e-01,  2.3703e-02,
          2.9727e-02,  4.2477e-02,  2.5634e-02,  1.9952e-03, -5.6919e-02,
         -2.7160e-02, -3.2904e-02,  6.6025e-02,  1.1901e-01, -4.5879e-02,
         -7.2621e-02, -3.2584e-02,  5.2341e-02,  4.5055e-02,  8.2530e-03,
          3.6702e-02, -1.3942e-02,  6.5392e-02, -2.6427e-02,  2.0639e-04,
         -1.3664e-02, -3.6281e-02, -1.9504e-02, -2.8974e-02,  3.9427e-02,
         -8.8409e-02,  2.6243e-03,  1.3671e-02,  4.8306e-02, -3.1157e-02,
         -1.1733e-01, -5.1169e-02, -8.8529e-02, -2.1896e-02,  1.4299e-02,
          4.4417e-02, -1.3482e-02,  7.4339e-02,  2.6638e-02, -1.9876e-02,
          1.7919e-02, -1.0605e-02, -9.0426e-02,  2.1327e-02,  1.4120e-01,
         -6.4718e

In [ ]:
# # 336358aa8a6be9682f49@cloudmailin.net

This is absolutely ridiculous. Our entire team (15 people) has been locked 
out for 3 hours now. We keep getting "500 Internal Server Error" when trying 
to login.

We're paying $299/month for Enterprise and this is UNACCEPTABLE. We have a 
client presentation in 1 hour and can't access any of our files.

Fix this IMMEDIATELY or we're canceling and switching to your competitor.

John Davis
CTO, TechStartup Inc.

Channel: Web Form
From: john.davis@techstartup.com
Subject: Your service is completely broken!!!

# 336358aa8a6be9682f49@cloudmailin.net

# https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2?inference_provider=hf-inference

https://www.cloudflare.com/application-services/products/bot-management/#bm-resources

https://developers.google.com/recaptcha/intro

https://developer.zendesk.com/documentation/ticketing/managing-tickets/improving-performance-by-creating-tickets-asynchronously/

support@none-55378.zendesk.com